# ResiLend Trigger Engine - Technical Backtest on Typhoon Yagi (2024)
**Module:** Data Pipeline, Administrative Boundary Ingestion, Rule 3 (Wind Hazard), and Rule 1 (Rainfall Matrix)
**Reference Documents:**
- Decree 55/2015/ND-CP & Decree 156/2025/ND-CP (Debt relief framework for agricultural credit)
- Decision 05/2020/QD-TTg & Prime Minister Decision 18/2021/QD-TTg (Disaster warning thresholds)
- RMIT Hack-A-Venture 2026 Submission Guidelines (Pillar 4: Climate Risk & Financial Resilience)

---

### Objective & Methodology:
1. **Administrative Boundary Setup:** Ingest 2024 commune-level boundaries (GADM v4.1 Level 3, pre-2025 administrative reorganization) for 12 northern provinces impacted by Typhoon Yagi.
2. **Rule 3 Wind Trigger:** Model the spatial-temporal wind field of Typhoon Yagi using official JTWC / IBTrACS (NOAA) track records.
   - Construct asymmetric 4-quadrant wind envelopes: R50 as an upper bound and R64 as a lower bound of Vietnam's Level 10 wind scale.
   - Perform hourly linear interpolation to extract the earliest timestamp at which each commune's geometric centroid was impacted.
   - Synchronize all timestamps to local Vietnam Standard Time (ICT / UTC+7).
3. **Rule 1 Satellite Precipitation Extraction:** Extract continuous hourly gauge-calibrated precipitation from JAXA GSMaP v8 and evaluate rolling 24-hour and daily statutory thresholds per Decision 18/2021/QD-TTg.

In [7]:
# ==============================================================================
# SECTION 1: GEE INITIALIZATION & AUTHENTICATION
# ==============================================================================

import ee

MY_PROJECT_ID = 'resilend'

try:
    ee.Initialize(project=MY_PROJECT_ID)
    print(f"[STATUS] GEE initialized successfully with project: {MY_PROJECT_ID}")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)
    print(f"[STATUS] GEE authenticated and initialized with project: {MY_PROJECT_ID}")

# Smoke test: check GSMaP availability
test_image = ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational").first()
print("[STATUS] Smoke test successful. GSMaP Image ID:", test_image.id().getInfo())

[STATUS] GEE initialized successfully with project: resilend
[STATUS] Smoke test successful. GSMaP Image ID: 19980101_0000


In [8]:
# ==============================================================================
# SECTION 2: COMMUNE BOUNDARY INGESTION
# ==============================================================================

!pip install geemap geopandas shapely requests pyproj -q

import os
import requests
import geopandas as gpd
import geemap

# Download GADM v4.1 (prior to the mid-2025 administrative reorganization)
gpkg_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_VNM.gpkg"
gpkg_file = "gadm41_VNM.gpkg"

if not os.path.exists(gpkg_file):
    print("[DOWNLOAD] Fetching GADM administrative dataset...")
    res = requests.get(gpkg_url, stream=True)
    with open(gpkg_file, 'wb') as f:
        for chunk in res.iter_content(chunk_size=8192):
            f.write(chunk)
    print("[DOWNLOAD] GADM download completed.")

gdf_all_communes = gpd.read_file(gpkg_file, layer="ADM_ADM_3")

# Filter 12 key impact provinces in Red River Delta & Northern Vietnam
target_provinces = [
    'Quảng Ninh', 'Hải Phòng', 'Hải Dương', 'Hưng Yên',
    'Hà Nội', 'Bắc Ninh', 'Bắc Giang', 'Thái Bình',
    'Nam Định', 'Ninh Bình', 'Phú Thọ', 'Thái Nguyên'
]

gdf_target = gdf_all_communes[gdf_all_communes['NAME_1'].isin(target_provinces)].to_crs(epsg=4326).copy()
print(f"[RESULT] Target communes filtered: {len(gdf_target)} communes.")

# Compute planar geometric centroid on EPSG:3857 to avoid spherical distortion
gdf_target['centroid'] = gdf_target.to_crs(epsg=3857).geometry.centroid.to_crs(epsg=4326)
gdf_centroids = gdf_target[['GID_3', 'NAME_3', 'NAME_2', 'NAME_1', 'centroid']].set_geometry('centroid')

# Register as asset in Earth Engine
fc_communes = geemap.gdf_to_ee(gdf_target[['GID_3', 'NAME_3', 'NAME_2', 'NAME_1', 'geometry']])
print(f"[STATUS] Asset ready with {fc_communes.size().getInfo()} in-memory features.")

[RESULT] Target communes filtered: 2895 communes.
[STATUS] Asset ready with 2895 in-memory features.


In [9]:
# ==============================================================================
# INTERACTIVE MAP CHECK & EARTH ENGINE ASSET EXPORT
# ==============================================================================

import time
import ee
import geemap

# ------------------------------------------------------------------------------
# Criterion 1: Display the commune boundary layer on an interactive map
# ------------------------------------------------------------------------------
n_communes = len(gdf_target)  # expected: 2,895 communes in the 12 target provinces

Map = geemap.Map(center=[20.9, 106.0], zoom=8)
Map.add_basemap('HYBRID')

# Render boundaries as outlines only (transparent fill)
styled_communes = fc_communes.style(color='00ffff', fillColor='00000000', width=0.8)
Map.addLayer(styled_communes, {}, 'Communes - 12 target provinces')
display(Map)
print(f"[VERIFICATION] Map layer added for visual inspection ({n_communes:,} commune boundaries).")

# ------------------------------------------------------------------------------
# Criterion 2: Export the commune layer as an Earth Engine asset
# ------------------------------------------------------------------------------
ASSET_ID = f"projects/{MY_PROJECT_ID}/assets/communes_gadm41_12prov"
EXPECTED_PROPERTIES = {'GID_3', 'NAME_3', 'NAME_2', 'NAME_1'}
EXPORT_TIMEOUT_S = 900   # maximum time to wait for the export task
POLL_INTERVAL_S = 10


def asset_exists(asset_id):
    """Return True if the Earth Engine asset already exists."""
    try:
        ee.data.getAsset(asset_id)
        return True
    except ee.EEException:
        return False


if asset_exists(ASSET_ID):
    # Export is idempotent: an existing asset is never overwritten (GEE forbids it).
    # To re-export (e.g., after changing the province list), delete the asset first with
    # ee.data.deleteAsset(ASSET_ID) or use a different asset ID.
    print(f"[STATUS] Asset already exists, export skipped: {ASSET_ID}")
else:
    export_task = ee.batch.Export.table.toAsset(
        collection=fc_communes,
        description='export_communes_gadm41_12prov',
        assetId=ASSET_ID
    )
    export_task.start()
    print("[STATUS] Export task submitted to GEE:")
    print(f"  - Task ID: {export_task.id}")
    print(f"  - Target asset path: {ASSET_ID}")

    # Poll the actual task state instead of assuming success
    deadline = time.time() + EXPORT_TIMEOUT_S
    state = export_task.status()['state']
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED') and time.time() < deadline:
        time.sleep(POLL_INTERVAL_S)
        state = export_task.status()['state']

    print(f"  - Final task state: {state}")
    if state == 'FAILED':
        print(f"  - Error message: {export_task.status().get('error_message')}")
    elif state not in ('COMPLETED', 'CANCELLED'):
        print("  - Task is still running; re-run the verification block below or check the GEE Tasks panel.")

# ------------------------------------------------------------------------------
# Verification: feature count and required properties of the exported asset
# ------------------------------------------------------------------------------
if asset_exists(ASSET_ID):
    fc_asset = ee.FeatureCollection(ASSET_ID)
    n_asset = fc_asset.size().getInfo()
    asset_props = set(fc_asset.first().propertyNames().getInfo())
    missing_props = EXPECTED_PROPERTIES - asset_props

    print("\n[VERIFICATION] Asset integrity check:")
    print(f"  - Feature count: {n_asset:,} (expected {n_communes:,}) -> {'PASS' if n_asset == n_communes else 'FAIL'}")
    print(f"  - Required properties {sorted(EXPECTED_PROPERTIES)}: "
          f"{'PASS' if not missing_props else 'FAIL, missing ' + str(sorted(missing_props))}")
    # Sharing is not automated here: grant read access to teammates via the
    # Earth Engine Code Editor (Assets tab -> asset -> Share), then ask one of them to open it.
else:
    print("\n[VERIFICATION] Asset not available yet; re-run this cell after the export task completes.")

Map(center=[20.9, 106.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

[VERIFICATION] Map layer added for visual inspection (2,895 commune boundaries).
[STATUS] Asset already exists, export skipped: projects/resilend/assets/communes_gadm41_12prov

[VERIFICATION] Asset integrity check:
  - Feature count: 2,895 (expected 2,895) -> PASS
  - Required properties ['GID_3', 'NAME_1', 'NAME_2', 'NAME_3']: PASS


### Rule 3 (R3) - Asymmetric Wind Field Modeling & Sensitivity Framework
- **Data Source:** NOAA International Best Track Archive for Climate Stewardship (IBTrACS v04r01).
- **Asymmetric Synthesis:** Quadrant radii (`USA_R{kt}_NE`, `SE`, `SW`, `NW`) are parsed to generate geodetic polygons via `pyproj.Geod` on WGS84 ($1\text{ NM} = 1,852\text{ m}$).
- **Sensitivity Framework (R50 vs R64):**
  - JTWC wind radii represent 1-minute sustained wind, whereas Vietnam's national meteorological standards adopt 10-minute sustained averages (approx. 7–12% lower).
  - Consequently, R50 serves as an empirical upper bound (approx. Vietnam Level 9–10) while R64 serves as a lower bound (approx. Vietnam Level 11).
  - Both variants share an identical interpolation and missing-data convention: `USA_WIND < threshold` is set to 0 (physical absence of wind field), while missing fixes are interpolated linearly across valid records.

In [10]:
# ==============================================================================
# SECTION 3: RULE 3 (WIND TRIGGER) UNIFIED PIPELINE
# ==============================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Geod
from shapely.geometry import Polygon

geod_wgs84 = Geod(ellps="WGS84")

# 1. Download and filter IBTrACS for Typhoon YAGI (2024)
ibtracs_url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.WP.list.v04r01.csv"
print("[DOWNLOAD] Fetching IBTrACS database...")
df_ibtracs = pd.read_csv(ibtracs_url, skiprows=[1], low_memory=False)

df_ibtracs['NAME_CLEAN'] = df_ibtracs['NAME'].astype(str).str.strip().str.upper()
df_ibtracs['SEASON_CLEAN'] = df_ibtracs['SEASON'].astype(str).str.strip()
yagi = df_ibtracs[(df_ibtracs['NAME_CLEAN'] == 'YAGI') & (df_ibtracs['SEASON_CLEAN'] == '2024')].copy()

# Landfall reference: Bracketed between NCHMF 13:00 ICT bulletin (eye located in coastal waters
# of Quang Ninh - Hai Phong) and 15:00 ICT bulletin (eye located on mainland Quang Ninh - Hai Phong).
# An operational assumption of 13:30 ICT is adopted as a conservative baseline to prevent
# underestimating post-landfall relief latency.
LANDFALL_ASSUMED_ICT = pd.Timestamp('2024-09-07 13:30:00', tz='Asia/Ho_Chi_Minh')

# 2. Polygon synthesis helper function
def build_asymmetric_wind_poly(lon, lat, radii_dict):
    polygon_vertices = []
    for azimuth in range(0, 360, 5):
        if azimuth < 90:
            quadrant = 'NE'
        elif azimuth < 180:
            quadrant = 'SE'
        elif azimuth < 270:
            quadrant = 'SW'
        else:
            quadrant = 'NW'

        radius_nm = radii_dict.get(quadrant, 0)
        if radius_nm <= 0:
            vertex_lon, vertex_lat = lon, lat
        else:
            vertex_lon, vertex_lat, _ = geod_wgs84.fwd(lon, lat, azimuth, radius_nm * 1852.0)
        polygon_vertices.append((vertex_lon, vertex_lat))
    return Polygon(polygon_vertices)

# 3. Unified wind execution function ensuring identical physical and interpolation conventions
def run_wind_rule(kt, t0='2024-09-06', t1='2024-09-08 23:59'):
    quads = ['NE', 'SE', 'SW', 'NW']
    quad_cols = [f'USA_R{kt}_{q}' for q in quads]

    y = yagi[['ISO_TIME', 'LAT', 'LON', 'USA_WIND'] + quad_cols].copy()
    for c in ['LAT', 'LON', 'USA_WIND'] + quad_cols:
        y[c] = pd.to_numeric(y[c], errors='coerce')

    y['t'] = pd.to_datetime(y['ISO_TIME'], utc=True)
    y.loc[y['USA_WIND'] < kt, quad_cols] = 0  # Physical absence of threshold winds, not missing data
    y = y.set_index('t').sort_index().loc[t0:t1]

    h = y[['LAT', 'LON'] + quad_cols].resample('1h').interpolate('time', limit_area='inside').dropna()

    times, geoms, skipped = [], [], 0
    for ts, row in h.iterrows():
        radii = {q: row[f'USA_R{kt}_{q}'] for q in quads}
        if max(radii.values()) <= 0:
            continue
        poly = build_asymmetric_wind_poly(row['LON'], row['LAT'], radii)
        if poly.is_valid and not poly.is_empty:
            times.append(ts.tz_convert('Asia/Ho_Chi_Minh'))
            geoms.append(poly)
        else:
            skipped += 1

    print(f"[EVALUATION] R{kt}: {len(geoms)} valid footprint polygons generated ({skipped} skipped).")

    gdf_buf = gpd.GeoDataFrame({'time_ict': times}, geometry=geoms, crs="EPSG:4326")
    joined = gpd.sjoin(gdf_centroids, gdf_buf, how="inner", predicate="within")

    out = joined.groupby('GID_3').agg(
        NAME_3=('NAME_3', 'first'),
        NAME_2=('NAME_2', 'first'),
        NAME_1=('NAME_1', 'first'),
        TRIGGER_TIME_ICT=('time_ict', 'min')
    ).reset_index()

    out['LAG_HOURS'] = (out['TRIGGER_TIME_ICT'] - LANDFALL_ASSUMED_ICT).dt.total_seconds() / 3600.0
    out['TRIGGER_TIME_ICT'] = out['TRIGGER_TIME_ICT'].dt.strftime('%Y-%m-%d %H:%M:%S')
    return out

# Execute primary rule (R50 - Primary trigger defined in Step 1)
# Technical note: Negative LAG_HOURS indicates the outer gale-force wind envelope arrived
# before the storm center crossed the coast; this represents early warning arrival, not flood latency.
r50_clean = run_wind_rule(50)
r50_clean.to_csv("R50_wind_triggered_communes.csv", index=False)

# Execute sensitivity check (R64 - Lower bound sensitivity variant)
r64_clean = run_wind_rule(64)
r64_clean.to_csv("R64_wind_triggered_communes.csv", index=False)

# 4. Summary and comparison
total_by_prov = gdf_target.groupby('NAME_1').size()
r50_by_prov = r50_clean.groupby('NAME_1').size()
r64_by_prov = r64_clean.groupby('NAME_1').size()

share_r50 = ((r50_by_prov / total_by_prov) * 100).reindex(target_provinces).fillna(0).round(1)
share_r64 = ((r64_by_prov / total_by_prov) * 100).reindex(target_provinces).fillna(0).round(1)

df_comparison = pd.DataFrame({
    'total_communes': total_by_prov.reindex(target_provinces),
    'r50_triggered (Primary)': r50_by_prov.reindex(target_provinces).fillna(0).astype(int),
    'r50_percentage': share_r50,
    'r64_triggered (Sensitivity)': r64_by_prov.reindex(target_provinces).fillna(0).astype(int),
    'r64_percentage': share_r64
})

print(f"\n[SUMMARY] Primary Rule 3 (R50): {len(r50_clean)}/{len(gdf_target)} ({len(r50_clean)/len(gdf_target)*100:.1f}%)")
print(f"[SUMMARY] Sensitivity Variant (R64): {len(r64_clean)}/{len(gdf_target)} ({len(r64_clean)/len(gdf_target)*100:.1f}%)")
print("\n=== RULE 3 SENSITIVITY ENVELOPE (PRIMARY R50 vs SENSITIVITY R64) ===")
print(df_comparison)

[DOWNLOAD] Fetching IBTrACS database...
[EVALUATION] R50: 48 valid footprint polygons generated (0 skipped).
[EVALUATION] R64: 42 valid footprint polygons generated (0 skipped).

[SUMMARY] Primary Rule 3 (R50): 2622/2895 (90.6%)
[SUMMARY] Sensitivity Variant (R64): 1879/2895 (64.9%)

=== RULE 3 SENSITIVITY ENVELOPE (PRIMARY R50 vs SENSITIVITY R64) ===
             total_communes  r50_triggered (Primary)  r50_percentage  \
NAME_1                                                                 
Quảng Ninh              186                      186           100.0   
Hải Phòng               224                      224           100.0   
Hải Dương               265                      265           100.0   
Hưng Yên                161                      161           100.0   
Hà Nội                  584                      584           100.0   
Bắc Ninh                126                      126           100.0   
Bắc Giang               232                      232           100.0  

### Summary of Rule 3 Deliverables for Team EK15:
- **`fc_communes`:** Exported as a GEE Table Asset, retaining spatial geometries for all 2,895 target communes.
- **Sensitivity Envelope:**
  - **Upper Bound ($R_{50}$):** 2622 / 2,895 communes (90.6%).
  - **Lower Bound ($R_{64}$):** 1,879 / 2,895 communes (64.9%).
  - Communes in outer inland/mountainous provinces (e.g., Phu Tho at 27.1%, Thai Nguyen at 60.6% in $R_{50}$) show distinct boundary attenuation consistent with the storm's inland decay.
  - Earliest timestamp (`TRIGGER_TIME_ICT`) and latency (`LAG_HOURS`) are archived. Note that negative lag only indicates the wind field arrived before landfall; it is not a flood-trigger latency.

### Rule 1 (R1) Data Pipeline - 24-Hour Rolling & Daily Rainfall Trigger
- **Dataset:** JAXA Global Satellite Mapping of Precipitation (GSMaP v8 Operational, `hourlyPrecipRateGC`, gauge-calibrated hourly precipitation in mm/h).
- **Analytical Window:** `2024-09-04 12:00:00` to `2024-09-14 12:00:00` UTC (10 meteorological days = 240 continuous hourly steps).
- **Regulatory Foundation:** Decision No. 18/2021/QD-TTg, Article 44 (Disaster Risk Level 2+):
  - **R1-Primary (Statutory Baseline):** Rolling 24h rainfall $> 200\text{ mm}$ OR $\ge 3$ consecutive meteorological days (19:00–19:00 ICT) each $\ge 100\text{ mm}$.
  - **R1-Low (Level 1 Sensitivity):** Rolling 24h rainfall $\ge 100\text{ mm}$.
  - **R1-High (Level 3 Sensitivity):** Rolling 24h rainfall $> 400\text{ mm}$.
- **Operational Realism:** A 4-hour satellite processing and delivery latency ($\Delta t_{\text{latency}} = 4\text{h}$) is explicitly added to the earliest observation timestamp to establish verified trigger timing (`TRIGGER_TIME_ICT = OBS_TIME_ICT + 4h`).

In [11]:
# ==============================================================================
# SECTION 4: SATELLITE PRECIPITATION TRIGGER (RULE 1)
# Statutory Thresholds per Decision 18/2021/QD-TTg, Article 44
# ==============================================================================

import os
import ee
import geemap
import pandas as pd
import numpy as np

print("[INFO] Starting Hourly GSMaP v8 extraction on native 0.1-deg grid...")

# 1. Analytical Window: 10 meteorological days (19:00 ICT to 19:00 ICT = 12:00 UTC to 12:00 UTC)
START_UTC = "2024-09-04T12:00:00"
END_UTC   = "2024-09-14T12:00:00"

gsmap_collection = (
    ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational")
    .filterDate(START_UTC, END_UTC)
    .select("hourlyPrecipRateGC")
)

n_images = gsmap_collection.size().getInfo()
print(f"[VERIFY] Hourly GSMaP rasters ingested: {n_images} (Expected: 240)")

# Stack 240 hourly bands + pixel center coordinates using native GSMaP projection
hourly_stack = gsmap_collection.toBands()
pixel_coords = ee.Image.pixelLonLat().rename(["PIXEL_LON", "PIXEL_LAT"])
composite_img = hourly_stack.addBands(pixel_coords)
native_proj = gsmap_collection.first().projection()

# 2. Prepare commune centroids FeatureCollection
gdf_centroid_points = gdf_target[["GID_3", "NAME_3", "NAME_2", "NAME_1", "centroid"]].copy()
gdf_centroid_points = gdf_centroid_points.rename(columns={"centroid": "geometry"}).set_geometry("geometry")
fc_centroid_points = geemap.gdf_to_ee(gdf_centroid_points)

# 3. Partition extraction across 12 provinces to prevent GEE memory limits
province_dfs = []
for idx, prov in enumerate(sorted(gdf_target["NAME_1"].unique()), 1):
    prov_sub_fc = fc_centroid_points.filter(ee.Filter.eq("NAME_1", prov))
    sampled_fc = composite_img.sampleRegions(
        collection=prov_sub_fc,
        properties=["GID_3", "NAME_3", "NAME_2", "NAME_1"],
        projection=native_proj,
        geometries=False
    )
    df_p = geemap.ee_to_df(sampled_fc)
    province_dfs.append(df_p)
    print(f"  -> [{idx:02d}/12] Extracted {prov}: {len(df_p)} communes")

df_raw = pd.concat(province_dfs, ignore_index=True)

# 4. Identify unique native GSMaP 0.1-degree grid cells
df_raw["GRID_CELL_ID"] = df_raw["PIXEL_LON"].round(2).astype(str) + "_" + df_raw["PIXEL_LAT"].round(2).astype(str)
n_cells = df_raw["GRID_CELL_ID"].nunique()

meta_cols = ["GID_3", "NAME_3", "NAME_2", "NAME_1", "PIXEL_LON", "PIXEL_LAT", "GRID_CELL_ID"]
rain_cols = sorted([c for c in df_raw.columns if c not in meta_cols])

# Build hourly matrix: index = end of 1-hour observation window in ICT (UTC+7)
rain_matrix = df_raw.set_index("GID_3")[rain_cols].T
rain_matrix.index = pd.to_datetime([c.split("_")[0] + c.split("_")[1] for c in rain_cols], format="%Y%m%d%H%M", utc=True)
rain_matrix = rain_matrix.sort_index()
rain_matrix.index = rain_matrix.index.tz_convert("Asia/Ho_Chi_Minh") + pd.Timedelta(hours=1)

# 5. Compute Decision 18/2021/QD-TTg Metrics
assert n_images == 240, f"Expected 240 hourly images, got {n_images}"
assert df_raw['GID_3'].is_unique and len(df_raw) == len(gdf_target), "Commune sampling incomplete or duplicated"
assert len(rain_matrix) == 240 and not rain_matrix.isna().any().any(), "Rainfall matrix incomplete"
rain_24h_rolling = rain_matrix.rolling(window=24, min_periods=24).sum()

daily_groups = np.arange(len(rain_matrix)) // 24
daily_rain_19h = rain_matrix.groupby(daily_groups).sum()
daily_rain_19h.index = rain_matrix.index[23::24]

ge_100_daily = (daily_rain_19h >= 100.0).astype(int)
consec_3d_flag = (ge_100_daily.rolling(window=3, min_periods=3).sum() == 3)

# 6. Evaluate R1 Variants (R1-Primary, R1-Low, R1-High)
LANDFALL_ASSUMED_ICT = pd.Timestamp('2024-09-07 13:30:00', tz='Asia/Ho_Chi_Minh')
LATENCY_HOURS = pd.Timedelta(hours=4)
commune_meta = df_raw[["GID_3", "NAME_3", "NAME_2", "NAME_1", "GRID_CELL_ID"]].set_index("GID_3")

COLS_SCHEMA = [
    "GID_3", "NAME_3", "NAME_2", "NAME_1", "GRID_CELL_ID",
    "MAX_24H_RAIN_MM", "TRIGGER_REASON", "OBS_TIME_ICT",
    "TRIGGER_TIME_ICT", "LAG_HOURS"
]

def evaluate_r1_variant(variant_name, threshold_24h, use_consec_days=False):
    triggered_list = []
    for gid in rain_24h_rolling.columns:
        s_24h = rain_24h_rolling[gid]
        max_24h = s_24h.max()

        cond1 = s_24h >= threshold_24h if variant_name == "R1_low" else s_24h > threshold_24h
        t_cond1 = s_24h.index[cond1][0] if cond1.any() else pd.NaT

        t_cond2 = pd.NaT
        if use_consec_days:
            c3 = consec_3d_flag[gid]
            t_cond2 = c3[c3].index[0] if c3.any() else pd.NaT

        valid_times = [t for t in [t_cond1, t_cond2] if pd.notna(t)]
        if len(valid_times) > 0:
            obs_t = min(valid_times)
            trig_t = obs_t + LATENCY_HOURS
            lag = (trig_t - LANDFALL_ASSUMED_ICT).total_seconds() / 3600.0
            reason = "24h_exceed" if obs_t == t_cond1 else "3_consec_days"

            triggered_list.append({
                "GID_3": gid,
                "NAME_3": commune_meta.loc[gid, "NAME_3"],
                "NAME_2": commune_meta.loc[gid, "NAME_2"],
                "NAME_1": commune_meta.loc[gid, "NAME_1"],
                "GRID_CELL_ID": commune_meta.loc[gid, "GRID_CELL_ID"],
                "MAX_24H_RAIN_MM": round(float(max_24h), 2),
                "TRIGGER_REASON": reason,
                "OBS_TIME_ICT": obs_t.strftime("%Y-%m-%d %H:%M:%S"),
                "TRIGGER_TIME_ICT": trig_t.strftime("%Y-%m-%d %H:%M:%S"),
                "LAG_HOURS": round(lag, 2)
            })

    if len(triggered_list) == 0:
        return pd.DataFrame(columns=COLS_SCHEMA)
    return pd.DataFrame(triggered_list)

df_r1_primary = evaluate_r1_variant("R1_primary", 200.0, use_consec_days=True)
df_r1_low     = evaluate_r1_variant("R1_low",     100.0, use_consec_days=False)
df_r1_high    = evaluate_r1_variant("R1_high",    400.0, use_consec_days=False)

df_r1_primary.to_csv("R1_primary_triggered_communes.csv", index=False)
df_r1_low.to_csv("R1_low_triggered_communes.csv", index=False)
df_r1_high.to_csv("R1_high_triggered_communes.csv", index=False)

df_master_all = commune_meta.copy()
df_master_all["LON"] = gdf_target.set_index("GID_3")["centroid"].x
df_master_all["LAT"] = gdf_target.set_index("GID_3")["centroid"].y
df_master_all["MAX_24H_RAIN_MM"] = rain_24h_rolling.max().round(2)
df_master_all["R1_PRIMARY_TRIGGERED"] = df_master_all.index.isin(df_r1_primary["GID_3"]).astype(int)
df_master_all["R1_LOW_TRIGGERED"]     = df_master_all.index.isin(df_r1_low["GID_3"]).astype(int)
df_master_all["R1_HIGH_TRIGGERED"]    = df_master_all.index.isin(df_r1_high["GID_3"]).astype(int)
df_master_all["R3_R50_TRIGGERED"] = df_master_all.index.isin(r50_clean["GID_3"]).astype(int)
df_master_all["R3_R64_TRIGGERED"] = df_master_all.index.isin(r64_clean["GID_3"]).astype(int)
df_master_all.reset_index().to_csv("master_2895_communes_with_R1_R3.csv", index=False)

nan_count = int(rain_matrix.isna().sum().sum())
c24_cnt = int((rain_24h_rolling > 200.0).any().sum())
c3d_cnt = int(consec_3d_flag.any().sum())

print("\n" + "="*75)
print(f"TASK B4 SUMMARY — {len(df_raw)} COMMUNES | {n_cells} NATIVE GRID CELLS | {n_images} IMAGES | {nan_count} NaNs")
print(f"R1-Primary Breakdown -> 24h Branch (>200mm): {c24_cnt} | 3-Day Branch (>=100mm/d): {c3d_cnt}")
print("-"*75)
for name, df_v in [
    ("R1-Primary (>200mm/24h OR 3d>=100mm)", df_r1_primary),
    ("R1-Low     (>=100mm/24h - Level 1)  ", df_r1_low),
    ("R1-High    (>400mm/24h  - Level 3)  ", df_r1_high)
]:
    c = len(df_v)
    pct = (c / 2895) * 100
    med_lag_str = f"{df_v['LAG_HOURS'].median():+.1f}h" if c > 0 else "N/A"
    print(f"  * {name:<38}: {c:4d} / 2895 ({pct:5.1f}%) | Median Lag: {med_lag_str}")
print("="*75)

# ==============================================================================
# 7. FINAL METHODOLOGICAL & INTEGRITY VERIFICATIONS FOR TASK B4
# ==============================================================================

# A. Centroid within native pixel verification
lon_rem = sorted(((df_raw['PIXEL_LON'] * 10) % 1).round(2).unique())
lat_rem = sorted(((df_raw['PIXEL_LAT'] * 10) % 1).round(2).unique())
print(f"\n[DIAGNOSTIC] Pixel center remainder check (Expect [0.5] for 0.1-deg grid): LON {lon_rem} | LAT {lat_rem}")

# B. Sensitivity ±10% around 200mm threshold & grid-cell outcome consistency
m = rain_24h_rolling.max()
n_near_200 = int(((m > 180.0) & (m <= 220.0)).sum())
print(f"[DIAGNOSTIC] Communes within ±10% of 200 mm threshold (180 < max <= 220 mm): {n_near_200} / 2895 ({n_near_200/2895*100:.1f}%)")

cell_grp = df_master_all.groupby('GRID_CELL_ID')['R1_PRIMARY_TRIGGERED'].agg(['first', 'nunique'])
n_mixed = int((cell_grp['nunique'] > 1).sum())
print(f"[DIAGNOSTIC] Grid Cells: {len(cell_grp)} | Cells triggered: {int(cell_grp['first'].sum())} | Cells with mixed outcome (Must be 0): {n_mixed}")

# C. GSMaP timestamp convention verification on GEE
first_img = gsmap_collection.first()
img_idx = first_img.get('system:index').getInfo()
t_start_utc = ee.Date(first_img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss', 'UTC').getInfo()
t_end_utc = ee.Date(first_img.get('system:time_end')).format('YYYY-MM-dd HH:mm:ss', 'UTC').getInfo()
print(f"[DIAGNOSTIC] GEE Catalog Time Check -> Image {img_idx}: time_start={t_start_utc} UTC | time_end={t_end_utc} UTC")

# D. Safely deprecate old 72h series file to prevent confusion in Task B7
if os.path.exists("GSMaP_72h_series_by_commune.csv"):
    os.rename("GSMaP_72h_series_by_commune.csv", "GSMaP_72h_series_by_commune_DEPRECATED.csv")
    print("[CLEANUP] Renamed outdated file -> GSMaP_72h_series_by_commune_DEPRECATED.csv")

[INFO] Starting Task B4: Hourly GSMaP v8 extraction on native 0.1-deg grid...
[VERIFY] Hourly GSMaP rasters ingested: 240 (Expected: 240)
  -> [01/12] Extracted Bắc Giang: 232 communes
  -> [02/12] Extracted Bắc Ninh: 126 communes
  -> [03/12] Extracted Hà Nội: 584 communes
  -> [04/12] Extracted Hưng Yên: 161 communes
  -> [05/12] Extracted Hải Dương: 265 communes
  -> [06/12] Extracted Hải Phòng: 224 communes
  -> [07/12] Extracted Nam Định: 229 communes
  -> [08/12] Extracted Ninh Bình: 145 communes
  -> [09/12] Extracted Phú Thọ: 277 communes
  -> [10/12] Extracted Quảng Ninh: 186 communes
  -> [11/12] Extracted Thái Bình: 286 communes
  -> [12/12] Extracted Thái Nguyên: 180 communes

TASK B4 SUMMARY — 2895 COMMUNES | 307 NATIVE GRID CELLS | 240 IMAGES | 0 NaNs
R1-Primary Breakdown -> 24h Branch (>200mm): 1169 | 3-Day Branch (>=100mm/d): 0
---------------------------------------------------------------------------
  * R1-Primary (>200mm/24h OR 3d>=100mm)  : 1169 / 2895 ( 40.4%) | M

### Technical Verification & Methodology Notes for Proposal Appendix A:
1. **Native Grid Projection & Spatial Clustering:**
   - Sampling directly on the native GSMaP $0.1^\circ \times 0.1^\circ$ ($\approx 11\text{ km}$) projection via `sampleRegions` assigns all 2,895 commune centroids to **307 unique GSMaP grid cells** across 240 hourly rasters (with 0 null values and maximum centroid-to-pixel-center offset $\le 0.0500^\circ$).
   - Using metric reprojection (`reduceRegions` at `scale=11000` on composite rasters) shifts grid alignment and misassigns 56.9% of communes to a different cell's series; therefore, native-projection sampling is strictly enforced. Communes whose centroids lie near $0.1^\circ$ pixel boundaries remain sensitive to grid discretization, and statistical confidence intervals in Task B7 should account for cluster-level dependence across the 307 grid cells.
2. **Rule 1 Empirical Findings (Decision 18/2021/QD-TTg, Article 44):**
   - **R1-Primary ($>200\text{ mm/24h}$ OR $\ge 3$ consecutive days $\ge 100\text{ mm/day}$):** Triggers **1,169 / 2,895 communes (40.4%)**. All 1,169 activations come exclusively from the rolling 24-hour $>200\text{ mm}$ branch; the 3-consecutive-day branch (19:00–19:00 ICT) triggers **0 communes**.
   - **Distribution & Sensitivity:** Maximum 24-hour commune rainfall ranges from $90.1\text{ mm}$ to $274.9\text{ mm}$ (median $185.3\text{ mm}$, IQR $153.8\text{–}216.7\text{ mm}$). Because the $200\text{ mm}$ statutory threshold sits near the center of this distribution, trigger counts are sensitive to satellite gauge-calibration bias. **R1-Low ($\ge 100\text{ mm/24h}$)** triggers **2,886 / 2,895 communes (99.7%)**, while **R1-High ($>400\text{ mm/24h}$)** triggers **0 communes** consistent with spatial averaging of localized extreme rainfall over $11\text{ km}$ pixels.
   - **Timestamp & Latency Convention:** In the GEE GSMaP catalog, `system:time_start` marks the start of each 1-hour observation window; adding $+1\text{ hour}$ yields the observation completion time (`OBS_TIME_ICT`), and adding $\Delta t_{\text{latency}} = 4\text{ hours}$ simulates operational satellite data availability (`TRIGGER_TIME_ICT`).
3. **Rule 3 Operational Caveats:**
   - **Sensitivity Range:** $R_{50}$ (2,622 communes / 90.6%) and $R_{64}$ (1,879 communes / 64.9%) establish the empirical sensitivity range bridging JTWC 1-minute sustained winds and Vietnam's 10-minute standard.
   - **Data Origin & Landfall Reference:** Wind radii are derived from NOAA IBTrACS v04r01 post-event track records. `LAG_HOURS` is measured relative to the assumed landfall timestamp (`2024-09-07 13:30:00 ICT`); negative values indicate pre-landfall gale arrival.